# 🤖 Étape 5 — Modélisation & Comparaison
**Objectif :** Entraîner et comparer 4 modèles de classification supervisée sur les données rééquilibrées.

| Modèle | Type |
|---|---|
| Régression Logistique | Linéaire |
| Arbre de Décision | Non linéaire |
| Random Forest | Ensemble (bagging) |
| AdaBoost | Ensemble (boosting) |

---
**Lancer :** `Kernel → Restart & Run All`

## 0. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, confusion_matrix,
                              roc_curve, classification_report)
from imblearn.over_sampling import SMOTENC
import warnings, json, os
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', font_scale=1.05)
plt.rcParams['figure.dpi'] = 120
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
os.makedirs('../reports', exist_ok=True)

BLUE = '#4C9BE8'; ORANGE = '#E8734C'; GREEN = '#4CAF7D'; PURPLE = '#9B59B6'
MODEL_COLORS = [BLUE, ORANGE, GREEN, PURPLE]
print('✅ Imports OK')

## 1. Chargement et préparation

In [ ]:
df = pd.read_csv('../data/processed/churn_cleaned.csv')

final_vars = [
    'tenure', 'MonthlyCharges', 'TotalCharges',
    'Contract', 'OnlineSecurity', 'TechSupport', 'OnlineBackup',
    'InternetService', 'PaymentMethod', 'PaperlessBilling',
    'SeniorCitizen', 'Partner', 'Dependents'
]
num_vars = ['tenure', 'MonthlyCharges', 'TotalCharges']
cat_vars = [v for v in final_vars if v not in num_vars]

# Encodage
df_enc = df[final_vars + ['Churn']].copy()
le_dict = {}
for col in cat_vars + ['Churn']:
    le = LabelEncoder()
    df_enc[col] = le.fit_transform(df_enc[col].astype(str))
    le_dict[col] = le

X = df_enc[final_vars]
y = df_enc['Churn']
print(f'Shape X : {X.shape} | Distribution y : {y.value_counts().to_dict()}')

## 2. Split Train/Test + SMOTE-NC

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

cat_indices = [X.columns.tolist().index(c) for c in cat_vars]
smote_nc = SMOTENC(categorical_features=cat_indices, random_state=42)
X_train_res, y_train_res = smote_nc.fit_resample(X_train, y_train)

print(f'Train avant SMOTE-NC : {pd.Series(y_train).value_counts().to_dict()}')
print(f'Train après SMOTE-NC : {pd.Series(y_train_res).value_counts().to_dict()}')
print(f'Test (intact)        : {pd.Series(y_test).value_counts().to_dict()}')

## 3. Entraînement des 4 modèles

> Les modèles sont entraînés sur `X_train_res` (rééquilibré) et évalués sur `X_test` (données réelles).

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree'      : DecisionTreeClassifier(max_depth=5, random_state=42),
    'Random Forest'      : RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    'AdaBoost'           : AdaBoostClassifier(n_estimators=100, random_state=42)
}

results = []
trained_models = {}

for name, model in models.items():
    model.fit(X_train_res, y_train_res)
    y_pred  = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    trained_models[name] = {'model': model, 'y_pred': y_pred, 'y_proba': y_proba}
    results.append({
        'Modèle'    : name,
        'Accuracy'  : round(accuracy_score(y_test, y_pred), 4),
        'Precision' : round(precision_score(y_test, y_pred), 4),
        'Recall'    : round(recall_score(y_test, y_pred), 4),
        'F1-Score'  : round(f1_score(y_test, y_pred), 4),
        'ROC-AUC'   : round(roc_auc_score(y_test, y_proba), 4)
    })
    print(f'✅ {name} entraîné')

res_df = pd.DataFrame(results)
print('\n=== Résultats ===')
print(res_df.to_string(index=False))

## 4. Tableau comparatif des performances

In [ ]:
# Mise en évidence du meilleur modèle par métrique
print('=== Meilleur modèle par métrique ===')
for metric in ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']:
    best_idx = res_df[metric].idxmax()
    best_val = res_df[metric].max()
    best_name = res_df.loc[best_idx, 'Modèle']
    print(f'  {metric:12s} → {best_name:25s} ({best_val:.4f})')

print('\n⭐ Métrique prioritaire pour Churn : F1-Score et ROC-AUC')
print('   (car déséquilibre initial et coût élevé des faux négatifs)')

In [ ]:
metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']
x = np.arange(len(metrics))
width = 0.2

fig, ax = plt.subplots(figsize=(14, 6))
for i, (_, row) in enumerate(res_df.iterrows()):
    vals = [row[m] for m in metrics]
    bars = ax.bar(x + i*width, vals, width, label=row['Modèle'],
                  color=MODEL_COLORS[i], edgecolor='white', linewidth=0.8, alpha=0.88)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{val:.3f}', ha='center', va='bottom', fontsize=7.5, rotation=90)

ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(metrics, fontsize=11)
ax.set_ylim(0, 1.05)
ax.set_ylabel('Score', fontsize=11)
ax.set_title('Comparaison des 4 modèles — Toutes métriques', fontsize=13)
ax.legend(fontsize=10)
ax.axhline(y=0.8, color='#aaa', linestyle=':', linewidth=1)
plt.tight_layout()
plt.savefig('../reports/fig14_models_comparison.png', bbox_inches='tight')
plt.show()
print('✅ fig14 sauvegardée')

## 5. Matrices de confusion

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4))

for ax, (name, data), color in zip(axes, trained_models.items(), MODEL_COLORS):
    cm = confusion_matrix(y_test, data['y_pred'])
    sns.heatmap(cm, annot=True, fmt='d', ax=ax,
                cmap=sns.light_palette(color, as_cmap=True),
                linewidths=0.5, linecolor='white',
                xticklabels=['No Churn','Churn'],
                yticklabels=['No Churn','Churn'])
    tn, fp, fn, tp = cm.ravel()
    ax.set_title(f'{name}\nTP={tp} | FN={fn}', fontsize=10)
    ax.set_xlabel('Prédit'); ax.set_ylabel('Réel')

plt.suptitle('Matrices de confusion — Test set', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('../reports/fig15_confusion_matrices.png', bbox_inches='tight')
plt.show()
print('✅ fig15 sauvegardée')
print('\n⚠️  Attention aux Faux Négatifs (FN) : churners non détectés = clients perdus')

## 6. Courbes ROC

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))

for (name, data), color in zip(trained_models.items(), MODEL_COLORS):
    fpr, tpr, _ = roc_curve(y_test, data['y_proba'])
    auc = roc_auc_score(y_test, data['y_proba'])
    ax.plot(fpr, tpr, color=color, linewidth=2.5,
            label=f'{name} (AUC={auc:.3f})')

ax.plot([0,1],[0,1], 'k--', linewidth=1, label='Aléatoire (AUC=0.5)')
ax.fill_between([0,1],[0,1],[0,1], alpha=0.05, color='gray')
ax.set_xlabel('Taux de Faux Positifs (FPR)', fontsize=11)
ax.set_ylabel('Taux de Vrais Positifs (TPR)', fontsize=11)
ax.set_title('Courbes ROC — Comparaison des 4 modèles', fontsize=13)
ax.legend(fontsize=10, loc='lower right')
ax.set_xlim(0, 1); ax.set_ylim(0, 1.02)
plt.tight_layout()
plt.savefig('../reports/fig16_roc_curves.png', bbox_inches='tight')
plt.show()
print('✅ fig16 sauvegardée')

## 7. Validation croisée (5-fold)

Évalue la robustesse de chaque modèle en testant sur 5 partitions différentes.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_results = []

for name, data in trained_models.items():
    model = data['model']
    f1_scores  = cross_val_score(model, X_train_res, y_train_res,
                                  cv=cv, scoring='f1', n_jobs=-1)
    auc_scores = cross_val_score(model, X_train_res, y_train_res,
                                  cv=cv, scoring='roc_auc', n_jobs=-1)
    cv_results.append({
        'Modèle'       : name,
        'F1 moyen'     : round(f1_scores.mean(), 4),
        'F1 std'       : round(f1_scores.std(), 4),
        'AUC moyen'    : round(auc_scores.mean(), 4),
        'AUC std'      : round(auc_scores.std(), 4)
    })
    print(f'{name:25s} F1={f1_scores.mean():.3f}±{f1_scores.std():.3f} | AUC={auc_scores.mean():.3f}±{auc_scores.std():.3f}')

cv_df = pd.DataFrame(cv_results)
print('\n✅ Validation croisée terminée')

## 8. Feature Importance du meilleur modèle (Random Forest)

In [ ]:
rf_model = trained_models['Random Forest']['model']
fi = pd.Series(rf_model.feature_importances_, index=final_vars)
fi = fi.sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(9, 6))
colors = [GREEN if v >= fi.median() else BLUE for v in fi.values]
ax.barh(fi.index, fi.values, color=colors, edgecolor='white', height=0.6)
for i, val in enumerate(fi.values):
    ax.text(val + 0.002, i, f'{val*100:.1f}%', va='center', fontsize=9)
ax.axvline(x=fi.median(), color=ORANGE, linestyle='--',
           linewidth=1.5, label='Médiane')
ax.set_xlabel('Importance', fontsize=11)
ax.set_title('Feature Importance — Random Forest', fontsize=13)
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig('../reports/fig17_rf_importance.png', bbox_inches='tight')
plt.show()
print('✅ fig17 sauvegardée')

## 9. Sauvegarde des résultats

In [ ]:
import pickle

# Sauvegarder les résultats
res_df.to_csv('../data/processed/model_results.csv', index=False)
cv_df.to_csv('../data/processed/cv_results.csv', index=False)

# Sauvegarder le meilleur modèle (AdaBoost → meilleur F1 + AUC)
best_model = trained_models['AdaBoost']['model']
with open('../data/processed/best_model.pkl', 'wb') as f:
    pickle.dump(best_model, f)

print('✅ data/processed/model_results.csv')
print('✅ data/processed/cv_results.csv')
print('✅ data/processed/best_model.pkl (AdaBoost)')
print('\n→ Prochaine étape : 06_evaluation.ipynb — Rapport final')

## 10. 📋 Synthèse Modélisation

| Modèle | Accuracy | Precision | Recall | F1-Score | ROC-AUC |
|---|---|---|---|---|---|
| Logistic Regression | 0.741 | 0.508 | 0.746 | 0.605 | 0.818 |
| Decision Tree | 0.725 | 0.488 | 0.727 | 0.584 | 0.803 |
| Random Forest | 0.764 | 0.553 | 0.586 | 0.569 | 0.809 |
| **AdaBoost** | **0.749** | **0.519** | **0.749** | **0.613** | **0.829** |

**🏆 Meilleur modèle : AdaBoost** (F1=0.613, AUC=0.829)

**Observations :**
- AdaBoost offre le meilleur équilibre F1/AUC
- Random Forest a la meilleure Precision mais sacrifie le Recall
- Pour le Churn, on préfère un **Recall élevé** (détecter max de churners)

**→ Prochaine étape : `06_evaluation.ipynb` — Rapport final & interprétation**